# Programmable Photonic Transformations

Can a programmable 2×2 photonic circuit implement a target complex optical transformation, and how do component-level imperfections affect its accuracy and transmission?

## Optical Linear Transformations

Under the  `10_directional_couplers_and_coupled_mode_theory`, `11_mach_zehnder_interferometers`

### Optical fields as complex vectors

Each component represents a power-normalized complex modal amplitude, such that its squared magnitude corresponds to the optical power carried by the associated mode.
$$\mathbf{a}_\text{in} = \begin{bmatrix}
a_1 \\
a_2
\end{bmatrix} \quad \mathbf{a}_\text{out} = \begin{bmatrix}
b_1 \\
b_2
\end{bmatrix}$$

$$P_\text{in} = \mathbf{a}_\text{in}^\dagger \mathbf{a}_\text{in}$$

$$\mathbf{a}_\text{out} = U\mathbf{a}_\text{in}$$

lossless
$$U^\dagger U = I$$

so,
$$P_\text{out} = \mathbf{a}_\text{in}^\dagger U^\dagger U \mathbf{a}_\text{in} = P_\text{in}$$

Therefore, 
$$\Vert \mathbf{a}_\text{out} \Vert^2 = \Vert \mathbf{a}_\text{in} \Vert^2$$

$$C(\theta) = \begin{bmatrix}
\cos\theta & -i\sin\theta \\
-i\sin\theta & \cos\theta
\end{bmatrix}$$

$$P(\phi) = \begin{bmatrix}
1 & 0 \\
0 & e^{i\phi}
\end{bmatrix}$$

Ideally, $50:50$ coupler MZI is
$$U_\text{MZI}(\theta_1, \theta_2, \phi) = C(\theta_2)P(\phi)C(\theta_1) $$
$\theta_1 = \theta_2 = \frac{\pi}{4}$

An MZI that adjusts only a single internal relative phase can describe only a limited set of matrices. However, a $2 \times 2$ unitary matrix generally possesses four real degrees of freedom. If we disregard the global phase, there are three physically distinct degrees of freedom. Given this limitation, let us introduce additional phase shifters—beyond the internal phase—to control the relative phases at the input and output ports, thereby expanding the range of implementable transformations.

Let's define additional phase shifters:
$$D_\text{in}(\beta) = \begin{bmatrix}
1 & 0 \\
0 & e^{i\beta}
\end{bmatrix}$$

$$D_\text{out}(\alpha) = \begin{bmatrix}
1 & 0 \\
0 & e^{i\alpha}
\end{bmatrix}$$

By adding input and output phase shifters to an ideal 50:50 MZI, one can represent any $2\times2$ unitary matrix, up to a global phase. In this note, Ignore the global phase of the independent 2-port transformation.
ignore
then, we can get
$$U(\alpha, \phi, \beta) = D_\text{out}(\alpha)U_\text{MZI}(\phi)D_\text{in}(\beta)$$

when $U_t$ is unitary,
$$F_\text{shape} = \frac{|\mathrm{Tr}(U_t^\dagger U)|^2}{2\,\mathrm{Tr}(U^\dagger U)}$$

$$T_\text{avg} = \frac{\mathrm{Tr}(U^\dagger U)}{2}$$

and also,
$$U(\mathbf{p}) \approx e^{i\chi}U_t$$
$$\mathbf{p} = (\alpha, \phi, \beta)$$
This means that by varying controllable phases, a target matrix can be implemented—up to a global phase.

We need these indicators because optical losses can occur, even while the target complex matrix form is perfectly preserved.

$$U = \sqrt{\eta}\,e^{i\chi}U_t$$

$$F_\text{shape} = 1, \quad T_\text{avg} = \eta$$

### Model assumptions
- Linear, coherent, power-normalised two-mode system

- Ideal 50:50 directional couplers

- Independently controllable relative phase shifts

- No loss in the baseline unitary model

In [2]:
from src.verification import absolute_error, complex_error, relative_error, find_convergence_threshold
from src.waveguide import create_uniform_grid, slab_index_profile, second_derivative_matrix, slab_waveguide_operator, solve_eigenmodes, select_guided_modes
from src.coupler import double_slab_cell_average

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import math


COLORS = {
    "numerical": "#1F5A94",
    "numerical_alt": "#4C78A8",
    
    "analytical": "#D97706",
    "approximation": "#2E7D32",
    
    "error": "#7E57C2",
    "error_alt": "#B07AA1",

    "reference": "#6B7280",
    "reference_alt": "#9CA3AF"
}

figure_dir = Path("../figures/12_programmable_photonic_transformations")
figure_dir.mkdir(parents=True, exist_ok=True)


## Case 1 — Target Unitary Synthesis

Can a programmable $2 \times 2$ photonic circuit reproduce a prescribed complex unitary transformation using independently controllable phase shifts?

임의의 입력 복소 진폭 벡터에 대해 목표 행렬과 동일한 선형 변환을 구현할 수 있는가?

$$U(\alpha, \phi, \beta) = D_\text{out}(\alpha)U_\text{MZI}(\phi)D_\text{in}(\beta)$$

$$D_\text{in}(\beta) = \begin{bmatrix}
1 & 0 \\
0 & e^{i\beta}
\end{bmatrix}$$

$$D_\text{out}(\alpha) = \begin{bmatrix}
1 & 0 \\
0 & e^{i\alpha}
\end{bmatrix}$$

control variant
$$\mathbf{p} = (\alpha, \phi, \beta)$$


$$U(\alpha,\phi,\beta) = D_{\mathrm{out}}(\alpha)U_{\mathrm{MZI}}(\phi)D_{\mathrm{in}}(\beta)$$

$$U(\alpha,\phi,\beta) = \begin{bmatrix}
1 & 0\\0 & e^{i\alpha}
\end{bmatrix}\frac{1}{2}\begin{bmatrix}
1-e^{i\phi} & -i(1+e^{i\phi})\\
-i(1+e^{i\phi}) & e^{i\phi}-1
\end{bmatrix}\begin{bmatrix}
1 & 0\\
0 & e^{i\beta}
\end{bmatrix}
$$

$$U(\alpha,\phi,\beta) = \frac{1}{2}\begin{bmatrix}
1-e^{i\phi} & -i(1+e^{i\phi})e^{i\beta}\\
-i(1+e^{i\phi})e^{i\alpha} & (e^{i\phi}-1)e^{i(\alpha+\beta)}
\end{bmatrix}
$$

Using Euler's formula,

$$1-e^{i\phi}=-2ie^{i\phi/2}\sin\frac{\phi}{2}$$

$$1+e^{i\phi}=2e^{i\phi/2}\cos\frac{\phi}{2}$$

Therefore,

$$U(\alpha,\phi,\beta)=e^{i(\phi/2-\pi/2)}\begin{bmatrix}
\sin\frac{\phi}{2} & e^{i\beta}\cos\frac{\phi}{2}\\
e^{i\alpha}\cos\frac{\phi}{2} & -e^{i(\alpha+\beta)}\sin\frac{\phi}{2}
\end{bmatrix}
$$

$$U^\dagger (\alpha,\phi,\beta) = e^{-i(\phi/2-\pi/2)}\begin{bmatrix}
\sin\frac{\phi}{2} & e^{-i\alpha}\cos\frac{\phi}{2}\\
e^{-i\beta}\cos\frac{\phi}{2} & -e^{-i(\alpha+\beta)}\sin\frac{\phi}{2}
\end{bmatrix}$$

$$U^\dagger U = \begin{bmatrix}
\sin^2\frac{\phi}{2} + \cos^2\frac{\phi}{2} & e^{-i\beta}(\sin\frac{\phi}{2}\cos\frac{\phi}{2} - \sin\frac{\phi}{2}\cos\frac{\phi}{2}) \\
e^{-i\beta}(\sin\frac{\phi}{2}\cos\frac{\phi}{2} - \sin\frac{\phi}{2}\cos\frac{\phi}{2}) & \sin^2\frac{\phi}{2} + \cos^2\frac{\phi}{2}
\end{bmatrix}$$

$$U^\dagger U = \begin{bmatrix}
1 & 0 \\
0 & 1
\end{bmatrix}$$

$$U^\dagger U = I$$

**Goal**:

$$U_t = \frac{1}{\sqrt{2}}\begin{bmatrix}
1 & e^{i\pi / 3} \\
e^{i\pi / 4} & -e^{i7\pi / 12}
\end{bmatrix}$$

$$
U_t^{\dagger} = \frac{1}{\sqrt{2}}\begin{bmatrix}
1 & e^{-i\pi / 4} \\
e^{-i\pi / 3} & -e^{-i7\pi / 12}
\end{bmatrix}
$$

$$U_t^{\dagger}U_t = \frac{1}{2}\begin{bmatrix}
1+1 & 0 \\
0 & 1+1
\end{bmatrix}$$

$$U_t^{\dagger}U_t = \begin{bmatrix}
1 & 0 \\
0 & 1
\end{bmatrix}
$$

$$U_t^{\dagger}U_t = I$$

$$U(\alpha,\phi,\beta)=e^{i(\phi/2-\pi/2)}\begin{bmatrix}
\sin\frac{\phi}{2} & e^{i\beta}\cos\frac{\phi}{2}\\
e^{i\alpha}\cos\frac{\phi}{2} & -e^{i(\alpha+\beta)}\sin\frac{\phi}{2}
\end{bmatrix}
$$

$$U(\alpha,\phi,\beta)= U_t$$

$$\alpha = \frac{1}{4}, \quad \beta = \frac{1}{3}, \quad \phi = \frac{\pi}{2}$$